In [ ]:
# Cell 1: Setup
!pip install -q sentence-transformers vstash
!git clone https://github.com/stffns/vstash.git /content/vstash 2>/dev/null || (cd /content/vstash && git pull origin develop)
%cd /content/vstash

In [ ]:
# Cell 2: Generate triples from ALL 5 BEIR datasets
!PYTHONPATH=/content/vstash python -m experiments.rrf_training_pairs --datasets scifact nfcorpus fiqa scidocs arguana

In [ ]:
# Cell 3: Train MNRL v3 (all 5 datasets)
import json
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

model = SentenceTransformer('BAAI/bge-small-en-v1.5')

triples = []
with open('experiments/results/rrf_training_pairs.jsonl') as f:
    for line in f:
        triples.append(json.loads(line))

print(f'Loaded {len(triples)} triples')

examples = [InputExample(texts=[t['query'], t['positive']]) for t in triples]
loader = DataLoader(examples, shuffle=True, batch_size=64)
loss = losses.MultipleNegativesRankingLoss(model)

model.fit(
    train_objectives=[(loader, loss)],
    epochs=2,
    warmup_steps=50,
    optimizer_params={'lr': 3e-6},
    output_path='experiments/models/vstash-bge-mnrl-v3',
    show_progress_bar=True,
)
model.save('experiments/models/vstash-bge-mnrl-v3')
print('Training done.')

In [ ]:
# Cell 4: Upload to HuggingFace
!hf upload stffens/bge-small-rrf-v2 /content/vstash/experiments/models/vstash-bge-mnrl-v3 --commit-message 'v2: trained on 5 BEIR datasets'

In [ ]:
# Cell 5: Export ONNX
!pip install -q optimum[onnxruntime]
from sentence_transformers import SentenceTransformer
from optimum.exporters.onnx import main_export
from pathlib import Path
import shutil

local = 'experiments/models/vstash-bge-mnrl-v3'
out = 'experiments/models/vstash-bge-rrf-v2-onnx'
Path(out).mkdir(parents=True, exist_ok=True)

main_export(model_name_or_path=local, output=f'{out}/onnx', task='feature-extraction')
for f in ['tokenizer.json', 'tokenizer_config.json', 'vocab.txt', 'special_tokens_map.json', 'config.json']:
    src = Path(local) / f
    if src.exists():
        shutil.copy(src, Path(out) / f)
print('ONNX exported')

In [ ]:
# Cell 6: Upload ONNX to HF
!hf upload stffens/bge-small-rrf-v2 /content/vstash/experiments/models/vstash-bge-rrf-v2-onnx/onnx --commit-message 'Add ONNX export'
# Also upload model.onnx to root for FastEmbed compatibility
!hf upload stffens/bge-small-rrf-v2 /content/vstash/experiments/models/vstash-bge-rrf-v2-onnx/onnx/model.onnx --path-in-repo model.onnx --commit-message 'ONNX model at root'

In [ ]:
# Cell 7: Evaluate on all 5 BEIR datasets (full pipeline)
!cd /content/vstash && PYTHONPATH=/content/vstash python experiments/beir_tuned_full.py